# 2. Data Preprocessing

Social media text is noisy (links, @usernames, "soooo goooood", extra spaces).
We clean it **minimally** - transformer models were pre-trained on raw text, so
aggressive cleaning (lowercasing, removing stopwords/emojis) usually makes results
worse. We only:

1. replace links with `http`
2. replace @usernames with `@user` (privacy + matches TweetEval's own convention)
3. squeeze character floods: `soooo` becomes `soo`
4. normalise whitespace

**Emojis and hashtags are kept** - they carry the emotional signal.
The cleaned datasets are saved to `data/processed/`.

In [1]:
import os
import re
from datasets import load_from_disk

RAW, PROCESSED = "../data/raw", "../data/processed"
os.makedirs(PROCESSED, exist_ok=True)

def clean_text(text):
    text = re.sub(r"https?://\S+|www\.\S+", "http", text)   # links
    text = re.sub(r"@\w+", "@user", text)                     # usernames
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)              # soooo -> soo
    text = re.sub(r"\s+", " ", text)                          # whitespace
    return text.strip()

# quick demonstration
demo = "@JohnSmith99 I loooove this!! 😍 see https://shop.example.com/deal  #happy"
print("before:", demo)
print("after :", clean_text(demo))

before: @JohnSmith99 I loooove this!! 😍 see https://shop.example.com/deal  #happy
after : @user I loove this!! 😍 see http #happy


In [2]:
# apply the cleaning to every dataset and save the processed copies
for name in ["tweeteval_sentiment", "goemotions"]:
    out = os.path.join(PROCESSED, name)
    if os.path.exists(out):
        print(f"[skip] {name} already processed")
        continue
    ds = load_from_disk(os.path.join(RAW, name))
    ds = ds.map(lambda batch: {"text": [clean_text(t) for t in batch["text"]]},
                batched=True)
    ds.save_to_disk(out)
    print(f"[done] {name} -> {out}")

Map:   0%|          | 0/45615 [00:00<?, ? examples/s]

Map:   0%|          | 0/12284 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/45615 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/12284 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/43410 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5426 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5427 [00:00<?, ? examples/s]

[done] goemotions -> ../data/processed/goemotions


In [3]:
# before / after comparison on real tweets
raw = load_from_disk(os.path.join(RAW, "tweeteval_sentiment"))["train"]
proc = load_from_disk(os.path.join(PROCESSED, "tweeteval_sentiment"))["train"]

for i in range(3):
    print("RAW      :", raw[i]["text"][:110])
    print("PROCESSED:", proc[i]["text"][:110])
    print("-" * 80)

RAW      : "QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRe
PROCESSED: "QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRe
--------------------------------------------------------------------------------
RAW      : "Ben Smith / Smith (concussion) remains out of the lineup Thursday, Curtis #NHL #SJ"
PROCESSED: "Ben Smith / Smith (concussion) remains out of the lineup Thursday, Curtis #NHL #SJ"
--------------------------------------------------------------------------------
RAW      : Sorry bout the stream last night I crashed out but will be on tonight for sure. Then back to Minecraft in pc t
PROCESSED: Sorry bout the stream last night I crashed out but will be on tonight for sure. Then back to Minecraft in pc t
--------------------------------------------------------------------------------
